# 演習5 解答編 ―― スレッドセーフなキュー

> まず `ex05_bounded_queue.ipynb` を自分で解いてから読んでください。

## 発展課題1 の解答 ―― `bool pop(T&)` と `T pop()`

**待たない形（`bool pop(T& out)`）が要る場面**

- **表示のように「最新だけあればよい」処理。**
  新しいフレームが来ていれば描き、来ていなければ前のフレームのまま次へ進む。
  ここで待ってしまうと、画面が固まります
- **終了処理。** 「残っているものを全部吸い出して終わる」というとき、
  待つ形では最後に空になった瞬間に固まってしまいます
- **他にもやることがあるスレッド。** キューを見に来ただけで、
  なければ別の仕事に移りたい場合

**待つ形（`T pop()`）が要る場面**

- **パイプラインの中段。** 前の段からデータが来るまで、やることは何もありません。
  ここで待たずにポーリングすると、演習4-1 の無駄働きが発生します
- **正しさのために「必ず1個受け取る」必要がある場合。**
  失敗する可能性がないぶん、使う側のコードが単純になります

実用のキューは、たいてい**3つ**を用意します。

- 待つ `pop()`
- 待たない `try_pop(T&)`
- **時間を切って待つ** `pop(T&, timeout)`

3つ目が実は重要で、「基本は待つが、いつまでも待ち続けはしない」という形が書けます。
終了フラグを見に戻る隙ができるので、**安全に終われるキュー**にはほぼ必ず入っています
（演習9で扱います）。

## 発展課題2 の解答 ―― `lk.unlock();` を消したら

**正しさ ⇒ まったく変わりません。**
`unique_lock` はスコープを抜けるときに必ず解錠するので、鍵が開かないままになることはありません。

**速さ ⇒ 理屈のうえでは損をします。**

`lk.unlock()` を消すと、`can_pop_.notify_one()` は**まだ施錠されている状態**で呼ばれます。
演習3の発展課題4で見たとおり、起こされた側は鍵を取り直せずに、もう一度眠ることになります
（hurry up and wait）。

```
lk.unlock() あり : 起こす → 相手はすぐ鍵を取れる          （1往復）
lk.unlock() なし : 起こす → 相手は鍵を取れず眠る → 解錠 → また起こされる  （2往復）
```

ただし、演習3でも書いたとおり、**実測では差が出ないことがほとんど**です。
最近の実装はこの無駄を減らす工夫をしています。

「必ずこう書け」ではなく、**なぜそう書いてあるのかを説明できる**ようになれば十分です。

## 発展課題3 の解答 ―― `if (q.size() < 3) q.push(x);`

**期待どおりに動きません。3個を超えます。**

`size()` も `push()` も、それぞれは鍵で正しく守られています。
**問題は、その2つの「あいだ」です。**

```cpp
if (q.size() < 3)      // ← ここで鍵をかけ、答えを受け取り、鍵を開ける
                       // ★ この隙間に、他のスレッドが push できてしまう
    q.push(x);         // ← ここでまた鍵をかける
```

`size()` が返した「2個」という答えは、**返した瞬間から過去の情報**です。
`push` を呼ぶころには、もう3個になっているかもしれません。

作る係が2人いれば、2人とも「いま2個だ」と見てから、2人とも入れます。結果は4個です。
次のセルで確かめます。

In [ ]:
%%writefile ans05a.cpp
#include <iostream>
#include <thread>
#include <queue>
#include <mutex>
#include <atomic>
#include <vector>
using namespace std;

// push も size も、それぞれは鍵で正しく守られている
class SafeQueue {
public:
    void push(int v) {
        lock_guard<mutex> g(mtx_);
        q_.push(v);
        if (q_.size() > peak_) peak_ = q_.size();
    }
    bool pop(int& out) {
        lock_guard<mutex> g(mtx_);
        if (q_.empty()) return false;
        out = q_.front(); q_.pop();
        return true;
    }
    size_t size() const { lock_guard<mutex> g(mtx_); return q_.size(); }
    size_t peak() const { lock_guard<mutex> g(mtx_); return peak_; }
private:
    queue<int> q_;
    size_t peak_ = 0;
    mutable mutex mtx_;
};

SafeQueue q;
atomic<bool> stop{false};

// 「3個未満なら入れる」を、使う側で書いた場合
void producer() {
    for (int i = 0; i < 200000; i++) {
        if (q.size() < 3) q.push(i);        // ← 確かめてから入れる
    }
}

void consumer() {
    int v;
    while (!stop) q.pop(v);
}

int main() {
    thread c(consumer);
    vector<thread> ps;
    for (int k = 0; k < 4; k++) ps.emplace_back(producer);   // 作る係4人
    for (auto& t : ps) t.join();
    stop = true; c.join();
    cout << "「3個未満なら入れる」と書いたのに、実際に並んだ最大の個数 = "
         << q.peak() << " 個\n";
    return 0;
}

In [ ]:
!g++ -std=c++17 -pthread ans05a.cpp -o ans05a
!for i in 1 2 3; do ./ans05a; done

「3個未満なら入れる」と書いたのに、**4〜5個**並んでいます。

これは演習2の発展課題4で見た罠と、まったく同じものです。

> **「確かめてから使う」は壊れる。確かめることと使うことを、同じ鍵の中でやらなければならない。**

完成版の `push` を見てください。

```cpp
std::unique_lock<std::mutex> lk(mtx_);
can_push_.wait(lk, [this] { return q_.size() < capacity_; });   // 確かめる
q_.push(v);                                                     // 入れる
```

**確かめるのと入れるのが、1本の鍵の中でつながっています。** 隙間がありません。
だから容量が守られます。

この形には名前があります。**「確かめてから動く」（check-then-act）を分けてはいけない**。
並行処理のバグの、かなりの割合がこれです。

なお、`size()` は**まったく役に立たない**わけではありません。
「いまどれくらい混んでいるか」を**表示・記録する**のには使えます。
使ってはいけないのは、**その値をもとに判断して動く**ことです。

## 発展課題4 の解答 ―― 条件変数を1本にまとめたら

### `notify_one` の場合 ⇒ **止まります**

条件変数が1本だと、そこには**種類の違う待ち人が混ざります**。

- 「空でなくなるのを待っている人」（取り出したい）
- 「満杯でなくなるのを待っている人」（入れたい）

`notify_one()` は**誰が起きるかを選べません。** 起こしたい相手とは違う人が起きて、
「自分の条件はまだ偽だ」と分かってまた眠ってしまうと、**通知はそこで消えます。**
本当に起きるべきだった人は、誰にも起こされないまま残ります。

これを **lost wakeup（通知の取りこぼし）** と呼びます。

次のセルで確かめます。容量1・作る係2人・受け取る係1人という、
待ち人の種類が混ざる状況を作ります。最後の1本は止まるので、5秒で強制終了させます。

In [ ]:
%%writefile ans05b.cpp
#include <iostream>
#include <thread>
#include <vector>
#include <queue>
#include <mutex>
#include <condition_variable>
using namespace std;

enum Mode { TWO_CV, ONE_CV_ALL, ONE_CV_ONE };

template <typename T>
class Q {
public:
    Q(size_t cap, Mode m) : capacity_(cap), mode_(m) {}

    void push(const T& v) {
        unique_lock<mutex> lk(mtx_);
        cv_push().wait(lk, [this] { woke_++; return q_.size() < capacity_; });
        q_.push(v);
        lk.unlock();
        wake(cv_pop());
    }
    T pop() {
        unique_lock<mutex> lk(mtx_);
        cv_pop().wait(lk, [this] { woke_++; return !q_.empty(); });
        T v = q_.front(); q_.pop();
        lk.unlock();
        wake(cv_push());
        return v;
    }
    long woke() const { return woke_; }

private:
    // 2本モードでは別々の条件変数、1本モードでは同じものを返す
    condition_variable& cv_pop()  { return a_; }
    condition_variable& cv_push() { return (mode_ == TWO_CV) ? b_ : a_; }
    void wake(condition_variable& cv) {
        if (mode_ == ONE_CV_ALL) cv.notify_all(); else cv.notify_one();
    }
    queue<T> q_;
    size_t capacity_;
    Mode mode_;
    long woke_ = 0;
    mutex mtx_;
    condition_variable a_, b_;
};

void run(Mode m, const char* label) {
    Q<int> q(1, m);                       // 容量1、作る係2人、受け取る係1人
    cout << label << flush;
    vector<thread> ts;
    ts.emplace_back([&] { for (int i = 0; i < 10; i++) q.push(i); });
    ts.emplace_back([&] { for (int i = 0; i < 10; i++) q.push(i); });
    ts.emplace_back([&] { for (int i = 0; i < 20; i++) q.pop(); });
    for (auto& t : ts) t.join();
    cout << " 20個すべて処理できた。述語を評価した回数 = " << q.woke() << "\n" << flush;
}

int main() {
    run(TWO_CV,     "【条件変数2本 + notify_one（正しい形）】");
    run(ONE_CV_ALL, "【条件変数1本 + notify_all      】");
    run(ONE_CV_ONE, "【条件変数1本 + notify_one      】");
    cout << "ここには到達しない\n";
    return 0;
}

In [ ]:
!g++ -std=c++17 -pthread ans05b.cpp -o ans05b
!timeout 5 ./ans05b; echo "終了コード=$? （124 なら止まった）"

- **条件変数2本 + `notify_one`**（正しい形）⇒ 動く
- **条件変数1本 + `notify_all`** ⇒ 動く
- **条件変数1本 + `notify_one`** ⇒ **止まる**

### `notify_all` の場合 ⇒ 動きますが、無駄が増えます

全員起こせば、本当に進める人も必ず起きるので、取りこぼしは起きません。
**正しさは保てます。**

ただし演習4の発展課題2で見たとおり、待っている人が増えるほど
「起きたけれど自分の番ではなかった」人が増えます。
待ち人の種類が混ざっているぶん、その割合はさらに悪くなります。

### まとめ

> **待つ理由が2種類あるなら、条件変数も2本用意する。**

そうすれば `notify_one` で「起こしたい種類の人」だけを1人起こせます。
完成版のキューが `can_pop_` と `can_push_` の2本を持っているのは、このためです。

（鍵は1本のままでよいことにも注意してください。守っているデータは `q_` 1つだけです。
**鍵の本数は「守るデータの数」、条件変数の本数は「待つ理由の数」**で決まります。）

## 発展課題5 の解答 ―― 容量を 1 にしたら

**パイプラインは成立します。** ただし**余裕がまったくありません。**

容量1なら、作る側は「受け取る側が処理中の1個」の次を1個だけ持てます。
つまり2つの段が同時に動くことはできます。

```
容量1
Read    R1..R2..R3..R4      キューに1個置ければ次へ進める
Infer   ..I1..I2..I3..I4    その裏で1つ前を処理
```

問題は、**段の所要時間がばらついたとき**です。

```
容量1（Read が1回だけ遅れた）
Read    R1..R2......R3..R4
Infer   ..I1..I2....??..I3     ← 手が空いて止まる
```

キューに1個しか置けないので、**貯金ができません。** 前の段が一瞬遅れると、
次の段はすぐ手待ちになります。

容量を増やすと、速いときに作った分を貯めておけるので、
**遅れたときにその貯金で食いつなげます。**

```
容量4（同じ揺らぎでも止まらない）
Read    R1R2R3R4....R5R6      速いうちに貯めておく
Infer   ..I1..I2..I3..I4..I5  貯金があるので止まらない
```

つまり容量とは、**段どうしの揺らぎを吸収するクッション**です。

ちなみに **容量0** にすると、入れる側と取り出す側が必ずその場で出会うことになり、
パイプラインではなく**待ち合わせ**になります。段が同時に動くことはできません。

「では容量はいくつにすればよいのか」――
これが **演習6** のテーマです。

## 発展課題6 の解答 ―― 容量をあとから変えたい場合

`capacity_` も**共有データ**です。守り方は `q_` と同じで、**鍵の中で変更します。**

```cpp
void set_capacity(std::size_t n) {
    { std::lock_guard<std::mutex> g(mtx_); capacity_ = n; }
    can_push_.notify_all();          // ← これを忘れると固まる
}
```

大事なのは2行目です。

容量を**増やした**とき、「満杯だから」と眠っている人が何人もいるかもしれません。
その人たちの述語は、いま真になった可能性があります。
**しかし、誰も起こさなければ、彼らは永久に眠ったままです。**
これは演習4の発展課題5で見た「`notify` の書き忘れ」と同じ事故です。

> **述語に出てくる変数を変えたら、必ず対応する条件変数を起こす。**

これは `q_` に限った話ではない、という点が重要です。
`capacity_` も、終了フラグ `done` も、述語に登場する以上は同じ扱いになります。

`notify_one` ではなく `notify_all` を使う理由も、演習4の発展課題2の基準どおりです。
容量を3増やせば3人が進めるようになるかもしれません。
**何人進めるか分からないときは全員起こします。**

なお、容量を**減らした**場合、すでに入っているものを追い出すことはできません。
`q_.size()` がしばらく `capacity_` を超えたままになります。
述語は `q_.size() < capacity_` なので、その間は誰も入れられなくなるだけで、
壊れはしません ―― が、こういう「一時的に不変条件が崩れる」状態があること自体は、
設計として意識しておく必要があります。

---

## 参考：本番のプログラムでは

ハッカソンで読むキューも、いま組み立てたものとほぼ同じ形です。

- 鍵1本、条件変数2本（「取り出せる」用と「入れられる」用）
- `push` / `pop` の述語はどちらもラムダ式で `[this]` を捕獲している
- コンストラクタで容量を渡す

読むときは、次の3点だけ確かめれば構造は分かります。

1. **鍵は何本で、何を守っているか**
2. **条件変数は何本で、それぞれ何を待っているか**
3. **どの操作が、どの条件変数を起こしているか**

この3つが見えれば、あとは演習4・5でやったことの繰り返しです。